# Week 7 — Full pick-and-place and baselines

**Goal: complete the task, and test whether the connectome actually matters.**

Week 6's loop, extended to the whole task. One control step (40 ms); all code in `sim/pickplace.py`, `sim/network7.py`, `sim/week7.py`:

| | |
|---|---|
| **arm state → 8 signals** | week 5's four (cube − gripper x, y, z; gripper width) **plus** goal − cube x, y, gripper height and cube height (the week 5 decision: the grasp, carry and place phases need them) |
| **encoder** | push/pull Poisson rates on the 335 JO-C/E neurons, now 16 channels of about 21 neurons, grouped by week 5's *wiring* rule |
| **brain** | the network advances `brain_ms` (chosen in each network's timing sweep, default 40 ms) |
| **decoder → readout** | rates of **all descending neurons** over the last `window_ms` → ridge → **dx, dy, dz and grip** (grip > 0 opens) |
| **controller → physics** | week 4 IK controller, 40 ms of MuJoCo |

**Policies, all trained the same way** (DAgger: 4 rounds × 30 episodes, the same seeds; round 0 the expert drives, rounds 1–3 the policy drives; half of each round with 5 mm action noise):

| policy | what sits between the 8 signals and the action |
|---|---|
| **connectome** | encoder → male-CNS network (frozen) → all DNs → linear readout |
| **random network** | encoder → the connectome with every connection's target shuffled → all DNs → linear readout |
| **MLP** | the 8 noise-free signals → MLP (2 × 64 tanh) |
| linear (reference) | the 8 noise-free signals → linear readout |
| expert (reference) | the week 4 script |

**The random network** (`random_network` = week 5's `shuffle_sub`): the post-synaptic end of every connection is permuted across all connections. Every neuron keeps its exact in- and out-degree, every connection keeps its synapse count and sign, and sign follows the presynaptic neuron. So size, degree distributions and the excitatory/inhibitory ratio (of neurons, connections and synapses) are identical; Step 3 checks this. Same rules for both networks, each applied to its own wiring and activity: encoder channels from its own connections; w_scale matched so the same share of neurons fires (week 5's rule); the same readout pool; timing chosen by the same rule.

### What the sandbox already showed (no brain, real physics, 50 test seeds)
- **A linear readout on the 8 signals completes 0/50; the MLP completes 50/50.** Reaching was linear enough (week 6); the full task is not: "go to the cube" and "go to the goal" must switch on whether the cube is held. So with a linear readout, any success has to come from nonlinearity the network supplies. This is now the real test of the connectome.
- **Readout size matters.** A linear readout on 68 random nonlinear features of the signals completes 0–42%; on 300 it completes 100%. On a 3,500-neuron stand-in spiking network, 68 readout neurons never lifted the cube, while 600 lifted it 96% of the time. So the readout is **all descending neurons** for both networks, not week 5's planned 68. Offline, Step 4 also scores the 68 most informative.
- **Expect success rates near 0 for both networks.** The stand-in network and its shuffle both completed 0/50, but they differed clearly in how far they got: the structured one carried the cube to the zone much more often. So the table also reports **progress milestones** (lifted, carried to the zone), and the verdict uses them.

### Fixed before any brain run
- **Success** (week 4 physics test): cube in the zone, at rest, released, gripper 6 cm clear, held 10 steps (0.4 s), within **150 steps (6 s)**. The expert needs 49 steps (median).
- **Placement accuracy:** cube-to-target distance at the end (median over successes; also over every episode where the cube was lifted). **Time to complete:** the first step of the successful hold (median over successes).
- **Milestones:** *lifted* (cube centre 2 cm off the table), *carried to zone* (lifted cube over the 5 cm zone).
- **Stall stop:** if neither the gripper nor the cube moves 5 mm in 1 s, the episode stops as a failure. A static readout can settle where its motion cancels to zero; the expert never stands still that long.
- **Verdict:** connectome vs random network on the same 50 test seeds, milestone by milestone from hardest to easiest (success → zone → lifted). The answer is the first milestone with a significant paired difference (exact McNemar test, p < 0.05), in either direction. Placement accuracy and time are compared only if both networks succeed on at least 10 seeds.

**Optional stretch (PPO) is not included.** At about 0.2 s of laptop time per brain-driven control step, PPO's usual 10⁶ steps would take more than two days. The plan says to cut it first.

### Where to run: Colab with a GPU (recommended) or the laptop
| | how it runs | estimated time |
|---|---|---|
| **Colab, GPU** (`USE_GPU = True`) | every stage runs its episodes side by side on the GPU: a DAgger round (30), the test (50), the timing sweep (10), the encoder sweep (16 trials), the activity matching (12 at a time). Episodes that finish drop out of the batch, so the GPU only works on the ones still running. | **about 1 hour** (from week 3's GPU benchmark: 1.2 s of compute per simulated second per episode at batch 32, vs 4.6 s on the laptop). The **GPU check** cell measures the real speed first. |
| **laptop** (`USE_GPU = False`) | the numba engine from weeks 3–6, one episode at a time | about 3 hours; up to 5–6 if the timing sweep picks 80 ms |

One episode at a time on a GPU would be *slower* than the laptop (week 3: 10.9 s per simulated second, almost all fixed overhead per 0.1 ms step). The speed comes from batching only.

Every stage saves to `data/week7/` (on Drive, in Colab). With `RESUME = True`, a rerun skips finished stages and DAgger resumes after its last finished round, so a Colab disconnect costs at most one round. Set `QUICK = True` first for a smoke test (a few minutes on the GPU), in a separate folder (`data/week7_quick/`).

**What the GPU changes in the results:** the Poisson input of every episode comes from a hash of (episode seed, time step, input neuron), so it is the same on any engine and in any batch. In float64, the GPU engine gives the same spikes as numba except for rare last-digit rounding (the GPU adds synaptic inputs in a varying order). float32 (the default, faster) diverges spike by spike after a while but keeps the same rates (week 3: rate correlation 0.9998). So a GPU run and a laptop run are statistically equivalent, not identical.

### Setup

In [ ]:
import sys, subprocess, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    try:
        import brian2, numba, mujoco, imageio_ffmpeg
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'brian2', 'numba', 'mujoco',
                        'imageio', 'imageio-ffmpeg'], check=True)
        print("packages installed -> restarting runtime. When it reconnects, choose Runtime -> Run all again.")
        os.kill(os.getpid(), 9)
    os.environ.setdefault('MUJOCO_GL', 'egl')          # headless rendering for the video
except ImportError:
    IN_COLAB = False

from pathlib import Path
BRAIN = Path('/content/drive/MyDrive/arm_simulation/brain') if IN_COLAB else Path.home() / 'arm_simulation/brain'
DATA = BRAIN / 'data'
ARM = next((p for p in [BRAIN.parent / 'arm', BRAIN.parent / 'week4/arm'] if (p / 'panda_env.py').exists()), None)
sys.path.insert(0, str(BRAIN))
if ARM: sys.path.insert(0, str(ARM))

QUICK = False        # True: smoke test (3 short DAgger rounds, 10 test seeds, no encoder sweep)
RESUME = True        # skip stages whose results are already saved; DAgger resumes after its last round
RUN_RANDOM = True    # Step 3; set False to stop after the connectome and come back later
USE_GPU = True       # False: numba on the CPU, one episode at a time (the laptop path)
GPU_DTYPE = 'float32'   # 'float64' matches numba spike for spike (bar rare rounding) but is slower on most GPUs
MAX_BATCH = 64       # most episodes run side by side on the GPU (a whole DAgger round or the test)

# the Panda model (week 4 used commit 98a70c5); fetched next to arm/ if it is missing
MENAGERIE = (ARM.parent if ARM else BRAIN.parent) / 'mujoco_menagerie'
if not (MENAGERIE / 'franka_emika_panda').exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/google-deepmind/mujoco_menagerie.git', str(MENAGERIE)], check=True)
    subprocess.run(['git', '-C', str(MENAGERIE), 'checkout', '-q', '98a70c5'], check=True)

GPU = None
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            GPU = torch.cuda.get_device_name(0)
            print(f"GPU: {GPU}, {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB; torch {torch.__version__}")
        else:
            print("USE_GPU = True but this runtime has no GPU. In Colab: Runtime -> Change runtime type -> T4 GPU. "
                  "Continuing on the CPU (numba).")
    except ImportError:
        print("USE_GPU = True but torch is not installed here. Continuing on the CPU (numba).")

for p in ['sim/lif.py', 'sim/lif_numba.py', 'sim/interface.py', 'sim/readout.py', 'sim/closed_loop.py',
          'sim/pickplace.py', 'sim/network7.py', 'sim/batch7.py', 'sim/week7.py', 'sim/video7.py',
          'data/weights_male-cns_v1.0/meta.json', 'data/shortlist_week1.csv', 'data/wsyn_choice.json']:
    print(f"{p:45s} {'ok' if (BRAIN / p).exists() else 'MISSING'}")
print(f"{'week 4 arm code':45s} {ARM if ARM else 'MISSING - put panda_env.py and expert.py in ' + str(BRAIN.parent / 'arm')}")
if ARM:
    print(f"{'mujoco_menagerie next to arm/':45s} {'ok' if (ARM.parent / 'mujoco_menagerie/franka_emika_panda').exists() else 'MISSING'}")

### Load the connectome, the neuron groups and the arm

In [ ]:
import json, time, gc, pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore', module='brian2')
from sim.lif import WeightSet, peak_mem_gb
from sim.interface import wiring_channels
from sim.pickplace import SIGNALS8, SPEC8, PP_MAX_STEPS
from sim.network7 import random_network, network_stats, degrees_equal, wiring_labels
from sim.week7 import (run_references, run_network, reference_states, offline_compare, comparison_rows,
                       verdict, signal_video_eps, paired_diff)
from panda_env import PandaPickPlace, OBS_NAMES, TARGET_XY, ZONE_RADIUS
from expert import expert

T_START = time.perf_counter()
HOST = 'colab' if IN_COLAB else 'laptop'
OUT = DATA / ('week7_quick' if QUICK else 'week7'); OUT.mkdir(exist_ok=True)

ws = WeightSet(DATA / 'weights_male-cns_v1.0')
W = json.loads((DATA / 'wsyn_choice.json').read_text())['w_scale']
JO = ws.by_type(r'^JO-(C|E)', cls='cb_sensory')
shortlist = pd.read_csv(DATA / 'shortlist_week1.csv')
g2i = pd.Series(ws.neurons.index, index=ws.neurons.bodyId)
DN_cand = g2i.loc[shortlist.query("group == 'middle_descending'").bodyId].to_numpy()
ALL_DN = ws.idx(ws.neurons['cls'].astype(str).str.contains('descending', case=False))
if len(ALL_DN) == 0:
    print("no class containing 'descending' -> the readout pool falls back to the 68 DN candidates")
    ALL_DN = DN_cand
full = ws.subnetwork(np.arange(ws.N))
env = PandaPickPlace()

# seeds: none used in weeks 4-6 (week 6: 60,000-63,029, 70,000-70,029 and 90,000-90,049)
N_ROUNDS = 3 if QUICK else 4
N_PER_ROUND = 6 if QUICK else 30
CFG = dict(
    train_seeds=[list(range(64_000 + 1000 * r, 64_000 + 1000 * r + N_PER_ROUND)) for r in range(N_ROUNDS)],
    betas=[1.0] + [0.0] * (N_ROUNDS - 1),                     # round 0: the expert drives
    test_seeds=list(range(92_000, 92_000 + (10 if QUICK else 50))),
    cut_seeds=list(range(92_000, 92_000 + (3 if QUICK else 10))),   # feedback-cut control
    timing_seeds=list(range(82_000, 82_000 + (4 if QUICK else 10))),
    timing_settings=((40, (20, 40)), (80, (40, 80))),         # brain ms per control step: windows tried
    k=None,                                                   # readout: the whole pool (all DNs); 68 = week 5's plan
    w_ref=W, sweep=not QUICK, n_video=2,
    use_gpu=GPU is not None, dtype=GPU_DTYPE, max_batch=MAX_BATCH, cpu_batch=1)
print(f"male CNS: {ws.N:,} neurons, {len(full['w']):,} connections; w_scale {W}; JO-C/E {len(JO)}; "
      f"readout pool: {len(ALL_DN)} descending neurons (DN candidates {len(DN_cand)})")
print(f"8 signals: {', '.join(SIGNALS8)}")
print(f"DAgger {N_ROUNDS} rounds x {N_PER_ROUND} episodes; test {len(CFG['test_seeds'])} seeds; "
      f"episode limit {PP_MAX_STEPS} steps; results in {OUT}; peak memory {peak_mem_gb():.2f} GB")
print(f"engine: {'torch on ' + GPU + ' (' + GPU_DTYPE + '), up to ' + str(MAX_BATCH) + ' episodes side by side' if GPU else 'numba on the CPU, one episode at a time'}")

C = dict(blue='#2a78d6', orange='#eb6834', aqua='#1baf7a', yellow='#eda100', violet='#6250d6',
         red='#d03b3b', gray='#9a9893', ink='#0b0b0b', ink2='#52514e', grid='#e6e5e1')
POL_C = {'connectome': C['blue'], 'random network': C['orange'], 'MLP': C['aqua'],
         'linear on 8 signals': C['violet'], 'expert': C['gray']}
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 9, 'axes.titlesize': 10,
                     'axes.titleweight': 'bold', 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.edgecolor': C['ink2'], 'axes.labelcolor': C['ink2'], 'xtick.color': C['ink2'],
                     'ytick.color': C['ink2'], 'axes.grid': True, 'grid.color': C['grid'], 'grid.linewidth': 0.6,
                     'axes.axisbelow': True, 'legend.frameon': False, 'lines.linewidth': 2})

### GPU check (about 2 minutes; skipped on the CPU)

Before the long run, on the real network:
1. **Same results as the laptop engine?** 4 trials of 200 ms with identical input spikes, on the GPU and on numba. Reported: the share of neurons with identical spike counts and the rate correlation. Expect about 100% identical in float64 and a rate correlation above 0.99 in float32.
2. **How fast?** Seconds of compute per simulated second per episode, at batch sizes 1, 8, 32 and 64. Week 3 measured 10.9 at batch 1 and 1.2 at batch 32 on a T4. The laptop does 4.6.

In [ ]:
if GPU:
    from sim.batch7 import engine_check
    CHECK = engine_check(full, JO, ALL_DN, dtype=GPU_DTYPE)
    for k, v in CHECK.items():
        if k.startswith('match'):
            print(f"{k.replace('match_', 'GPU ')} vs numba: {v['identical_counts']:.2%} of active neurons with identical "
                  f"spike counts, rate correlation {v['rate_corr']:.4f} ({v['spikes_torch']:,} vs {v['spikes_numba']:,} spikes)")
    sp = CHECK['speed_s_per_trial_s']
    best = min(sp, key=sp.get)
    print(f"fastest: batch {best}, {sp[best]:.2f} s per simulated second per episode "
          f"({4.63 / sp[best]:.1f}x the laptop's 4.63)")
    if CHECK[f'match_{GPU_DTYPE}']['rate_corr'] < 0.99:
        print("WARNING: rates disagree with numba more than expected. Set GPU_DTYPE = 'float64' and rerun this cell.")
    gc.collect()
else:
    CHECK = None
    print("no GPU: skipped")

## Step 1: the policies without a brain (about 3 minutes, physics only)

The expert, then the **MLP baseline** and the linear reference, each trained with exactly the DAgger rounds and seeds the networks will get, and tested on the same 50 seeds. The sandbox gave linear 0/50 and MLP 50/50.

In [ ]:
REF = run_references(env, expert, OBS_NAMES, CFG, OUT, resume=RESUME)
for k in ('linear on 8 signals', 'MLP'):
    print(f"{k}: success while collecting, by round:", [f"{r['success']:.0%}" for r in REF[k]['log']])

## Step 2: the connectome (GPU: about 20–30 min; laptop: about 1.5 hours, or 2.7 if the timing rule picks 80 ms)

`run_network` does, in order:
1. **Encoder:** 16 channels (8 signals × push/pull) from week 5's wiring rule, computed on the network's own connections; saved as `encoder8_connectome.json`.
2. **Activity:** the connectome keeps its week 2 `w_scale`. The share of non-input neurons firing during a reference episode is measured; the random network will be matched to it.
3. **Encoder check** (the week 5 decision): tuning sweeps of the 8 signals, 2 seeds, held-out R² per signal from all DNs. About 4 min.
4. **Timing:** with the expert driving 10 episodes, the network runs at 40 ms and at 80 ms of brain time per 40 ms control step; windows of 20/40 ms (and 80 ms) are scored from the same spikes. The rule (`choose_timing`, fixed in advance): the best out-of-episode score wins, but 80 ms per step doubles the compute, so it must beat the best 40 ms setting by 0.05; anything within 0.01 of 40/40 keeps 40/40. The lag-1 score shows whether the network reports the state late.
5. **DAgger** (4 × 30 episodes), checkpointed after every round. **Test** on the 50 seeds (recorded, so the video shows real test episodes), then the **feedback-cut control** on 10 (the same brain and readout, but the brain only ever sees the first state).

First, a check that the encoder-channel code used here (which works on any network) gives week 5's split when given the connectome.

In [ ]:
lab_ws = wiring_channels(ws, JO, 16)
lab_sub = wiring_labels(full, JO, 16)
ws._out = None; gc.collect()                       # free WeightSet's cached adjacency
print(f"wiring channels, WeightSet version vs network version: {np.mean(lab_ws == lab_sub):.1%} of JO neurons "
      f"get the same channel (differences come only from edges of sign-0 neurons, which the network omits)")

S_REF = reference_states(env, expert, OBS_NAMES)
NETS = {}
NETS['connectome'] = run_network('connectome', full, JO, ALL_DN, env, expert, OBS_NAMES, CFG, OUT,
                                 w_scale=W, ref_states=S_REF, resume=RESUME, extra_record=DN_cand)
print(f"peak memory {peak_mem_gb():.2f} GB")

In [ ]:
def timing_table(res):
    t = pd.DataFrame(res['timing']['rows'])
    t = t.pivot_table(index=['brain_ms', 'window_ms'], columns='lag', values='score').rename(
        columns={0: 'score, lag 0', 1: 'score, lag 1 (state one step late)'})
    t['wall ms per control step'] = pd.DataFrame(res['timing']['rows']).groupby(['brain_ms', 'window_ms'])['wall_ms_per_step'].first()
    return t.round(3)
print("timing sweep (score = mean of dx, dy, dz R2 and grip skill; out-of-episode):")
display(timing_table(NETS['connectome']))
if 'sweep_r2' in NETS['connectome']:
    print("encoder check, held-out R2 per signal (tuning sweeps, all DNs):")
    display(pd.Series(NETS['connectome']['sweep_r2']).to_frame('connectome').T)

## Step 3: the random network (same as Step 2)

Built from the connectome by `random_network(full, seed=0)`. The table checks the match. Then the same `run_network`, with two differences that are the same *rule* applied to a different network: the encoder channels come from the random network's own wiring, and `w_scale` is searched (12 values at a time, then 12 more inside the bracket) until the same share of non-input neurons fires as in the connectome (week 5 needed 1.14 for this).

In [ ]:
if RUN_RANDOM:
    rnet = random_network(full, seed=0)
    st_c, st_r = network_stats(full), network_stats(rnet)
    keys = ['neurons', 'connections', 'synapses', 'exc_neurons', 'inh_neurons', 'exc_conn_frac', 'exc_syn_frac',
            'in_deg_median', 'in_deg_max', 'out_deg_median', 'out_deg_max', 'self_loops']
    display(pd.DataFrame({'connectome': [st_c[k] for k in keys], 'random network': [st_r[k] for k in keys]}, index=keys))
    print("every neuron keeps its exact in- and out-degree:", degrees_equal(st_c, st_r))
    del st_c, st_r
    NETS['random network'] = run_network('random network', rnet, JO, ALL_DN, env, expert, OBS_NAMES, CFG, OUT,
                                         match_target=NETS['connectome']['frac_firing'], ref_states=S_REF,
                                         resume=RESUME, extra_record=DN_cand)
    del rnet; gc.collect()
    display(timing_table(NETS['random network']))
    if 'sweep_r2' in NETS['random network']:
        display(pd.DataFrame({n: NETS[n]['sweep_r2'] for n in NETS}).T.rename_axis('held-out sweep R2'))
else:
    print("skipped (RUN_RANDOM = False). Set it to True and rerun: finished stages are loaded, not rerun.")

## Step 4: the comparison (deliverable)

Same 50 test seeds for every policy. Then the pre-registered verdict, the paired milestone tests, where each policy failed, the feedback-cut control, and an **open-loop** comparison: in DAgger round 0 the expert drove both networks through *identical* states (checked), so decoding there compares the two networks on exactly the same input.

In [ ]:
SUMM = {k: v['summary'] for k, v in NETS.items()}
SUMM['MLP'] = REF['MLP']['summary']
SUMM['linear on 8 signals'] = REF['linear on 8 signals']['summary']
SUMM['expert'] = REF['expert']['summary']
TABLE = pd.DataFrame(comparison_rows(SUMM)).set_index('policy')
pct = {c: '{:.0%}' for c in ('success rate', 'lifted', 'carried to zone', 'stalled')}
display(TABLE.style.format(pct, na_rep='-'))
TABLE.to_csv(OUT / f'week7_table_{HOST}.csv')

VERDICT, PM = (None, None)
if 'random network' in NETS:
    VERDICT, PM = verdict(NETS['connectome']['outs'], NETS['random network']['outs'])
    print("VERDICT:", VERDICT)
    display(pd.DataFrame(PM).T.rename(columns={'a': 'connectome', 'b': 'random network',
                                               'only_a': 'only connectome', 'only_b': 'only random'}).round(3))
    PD = paired_diff(NETS['connectome']['outs'], NETS['random network']['outs'])
    print(f"success rate, connectome - random: {PD['diff']:+.0%} (paired bootstrap 95% CI {PD['ci'][0]:+.0%} to {PD['ci'][1]:+.0%})")
    for k in ('connectome', 'random network'):
        s = SUMM[k]
        if s['success'] < 10:
            print(f"  {k}: {s['success']} successes, under 10, so its placement and time are not compared")

print("\nwhere each policy ended up (test episodes):")
display(pd.DataFrame({k: s['outcomes'] for k, s in SUMM.items()}).fillna(0).astype(int).T)
for k in NETS:
    if 'cut' in NETS[k]:
        c = NETS[k]['cut']['summary']
        print(f"feedback cut, {k}: success {c['success']}/{c['n']}, lifted {c['lifted_rate']:.0%}, "
              f"zone {c['zone_rate']:.0%}  (vs {SUMM[k]['lifted_rate']:.0%} lifted with feedback)")

In [ ]:
if len(NETS) > 1:
    OFF, SAME = offline_compare(NETS)
    print("round 0 states identical across networks:", SAME)
    rows = []
    for n, r in OFF.items():
        rows.append({'network': n, 'readout': 'all DNs', **r['action_all']})
        rows.append({'network': n, 'readout': '68 most informative', **r['action_k']})
    print("expert action decoded from the network's DNs, out-of-episode (round 0, identical states); "
          "grip as balanced accuracy:")
    display(pd.DataFrame(rows).set_index(['network', 'readout']))
    print("the 8 encoded signals decoded from all DNs (R2):")
    display(pd.DataFrame({n: r['signals'] for n, r in OFF.items()}).T)
else:
    OFF, SAME = None, None

## Step 5: what the policies did

- **Milestones** for every policy on the test seeds.
- **Cube paths from above** for the two networks: where the cube went, against the target zone.
- **Placement error** of every episode where the cube was lifted.
- **Learning curves:** success and lifted while collecting, per DAgger round.

In [ ]:
POLS = [k for k in ('connectome', 'random network', 'MLP', 'linear on 8 signals') if k in SUMM]
OUTS = {k: NETS[k]['outs'] for k in NETS}
OUTS.update({k: REF[k]['outs'] for k in ('MLP', 'linear on 8 signals', 'expert')})
fig, axs = plt.subplots(1, 4, figsize=(17.5, 4.2), gridspec_kw=dict(width_ratios=[1.25, 1, 1, 1]))
ax = axs[0]
ms = [('lifted_rate', 'lifted'), ('zone_rate', 'carried to zone'), ('success_rate', 'success')]
wd = 0.8 / len(POLS)
for j, p in enumerate(POLS):
    v = [SUMM[p][m] for m, _ in ms]
    ax.bar(np.arange(3) + (j - (len(POLS) - 1) / 2) * wd, v, wd * 0.92, color=POL_C[p], label=p)
ax.set_xticks(range(3), [l for _, l in ms]); ax.set_ylim(0, 1.05); ax.set_ylabel('share of test seeds')
ax.set_title('milestones'); ax.legend(fontsize=7, loc='upper right')

ax = axs[1]
for p in [k for k in ('connectome', 'random network') if k in OUTS]:
    for o in OUTS[p]:
        cube = o['obs'][:, 8:10] * 1000
        ax.plot(cube[:, 0], cube[:, 1], color=POL_C[p], lw=0.8, alpha=0.5)
    ax.plot([], [], color=POL_C[p], label=p)
ax.add_patch(plt.Circle(TARGET_XY * 1000, ZONE_RADIUS * 1000, fill=False, color=C['ink'], lw=1.2))
ax.set_aspect('equal'); ax.set_xlabel('cube x (mm)'); ax.set_ylabel('cube y (mm)')
ax.set_title('cube paths (circle: target zone)'); ax.legend(fontsize=7, loc='lower left')

ax = axs[2]
for j, p in enumerate(POLS + ['expert']):
    v = [o['place_error_mm'] for o in OUTS[p] if o['lifted']]
    ax.scatter(np.full(len(v), j) + np.random.default_rng(j).uniform(-0.18, 0.18, len(v)), v, s=9,
               color=POL_C[p], alpha=0.7)
ax.axhline(ZONE_RADIUS * 1000, color=C['ink2'], lw=0.8, ls='--')
ax.text(-0.4, ZONE_RADIUS * 1000 * 1.1, 'zone edge', fontsize=7, color=C['ink2'])
ax.set_yscale('log'); ax.set_xticks(range(len(POLS) + 1), [p.replace(' on 8 signals', '') for p in POLS + ['expert']],
                                    rotation=20)
ax.set_ylabel('cube to target at the end (mm)'); ax.set_title('placement error (lifted episodes)')

ax = axs[3]
logs = {k: NETS[k]['dagger']['log'] for k in NETS}
logs.update({k: REF[k]['log'] for k in ('MLP', 'linear on 8 signals')})
for p in POLS:
    x = np.arange(len(logs[p]))
    ax.plot(x, [r['success'] for r in logs[p]], marker='o', color=POL_C[p], label=f'{p}: success')
    ax.plot(x, [r['lifted'] for r in logs[p]], marker='.', ls=':', lw=1.3, color=POL_C[p])
ax.set_xticks(x, ['0\n(expert drives)'] + [str(i) for i in x[1:]]); ax.set_xlabel('DAgger round')
ax.set_ylim(-0.05, 1.05); ax.set_ylabel('while collecting'); ax.set_title('learning (dotted: lifted)')
ax.legend(fontsize=6.5, loc='center right')
fig.tight_layout(); fig.savefig(OUT / f'week7_summary_{HOST}.png', bbox_inches='tight'); plt.show()

## Step 6: the video

Each network's best two test episodes (recorded during the test itself), plus one MLP episode. Left: the arm. Right: spikes of the 100 readout neurons with the largest weights, and the decoded dx, dy, dz and grip against what the expert would have done. Half speed. Rendering needs OpenGL: on WSL, `sudo apt install libosmesa6`, or render `week7_episodes.pkl` on Colab (the last printed line says how).

In [ ]:
EPISODES = []
for k in NETS:
    for e in NETS[k]['video_eps']:
        print(f"{k}, seed {e['seed']}: {e['outcome']}")
        EPISODES.append(e)
EPISODES += signal_video_eps(env, expert, OBS_NAMES, REF['MLP']['readout'], 'MLP', CFG['test_seeds'][:1])
pickle.dump(EPISODES, open(OUT / 'week7_episodes.pkl', 'wb'))
from sim.video7 import render
VIDEO = OUT / f'week7_pick_place_{HOST}.mp4'
GL = render(str(OUT / 'week7_episodes.pkl'), str(VIDEO), slow=0.5)
if GL:
    from IPython.display import Video
    display(Video(str(VIDEO), embed=True, width=900))

## Step 7: results file

In [ ]:
def jsonable(s):
    return {k: (list(v) if isinstance(v, tuple) else v) for k, v in s.items()}
summary = dict(
    host=HOST, quick=QUICK, gpu=GPU, gpu_dtype=GPU_DTYPE if GPU else None, gpu_check=CHECK, w_scale_connectome=W, signals=SIGNALS8, readout='all descending neurons, ridge',
    readout_pool=int(len(ALL_DN)), dagger=dict(rounds=N_ROUNDS, episodes_per_round=N_PER_ROUND),
    table=TABLE.reset_index().to_dict('records'), verdict=VERDICT, paired_milestones=PM,
    outcomes={k: s['outcomes'] for k, s in SUMM.items()},
    networks={k: dict(w_scale=v['w_scale'], frac_firing=v['frac_firing'], pool_mean_hz=v['pool_mean_hz'],
                      pool_active=v['pool_active'], encoder_overlap=v['encoder_overlap'],
                      timing=v['timing']['choice'], timing_rows=v['timing']['rows'], sweep_r2=v.get('sweep_r2'),
                      dagger_log=v['dagger']['log'], alpha=v['dagger']['alpha'],
                      cut=v['cut']['summary'] if 'cut' in v else None, brain_s=v['brain_s'],
                      wall_min=v['wall_min'], engine=v.get('engine'),
                      brain_s_per_sim_s=v.get('brain_s_per_sim_s'),
                      stats={kk: vv for kk, vv in v['stats'].items() if kk not in ('in_deg', 'out_deg')})
              for k, v in NETS.items()},
    offline=OFF, round0_identical=SAME, video=str(VIDEO) if GL else None,
    wall_min=(time.perf_counter() - T_START) / 60)
(OUT / f'week7_results_{HOST}.json').write_text(json.dumps(summary, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else float(o)))
print(TABLE[['success', 'placement error, successes (mm, median)', 'time to complete (s, median)',
             'lifted', 'carried to zone']].to_string())
print("\nVERDICT:", VERDICT)
print(f"\nthis session {summary['wall_min']:.0f} min. Send me week7_results_{HOST}.json, week7_summary_{HOST}.png "
      f"and the video from {OUT}.")